In [ ]:
# If running within Google Colab it sets up the environment
# Please enable GPU under Runtime > Change runtime type

import os, sys

if 'google.colab' in sys.modules:
    repo_url = "https://github.com/mauro-m-monsalve/NeuralGeometry.git"
    repo_dir = "/content/NeuralGeometry"
    if not os.path.exists(repo_dir):
        !git clone {repo_url} {repo_dir}
    os.chdir(repo_dir)
    
    # # Optionally install dependencies not included in colab
    # !pip install pot

#
# Preprocessing and training of the LFADS model
#

This notebook demonstrates how the dataset was preprocessed for LFADS model training, using Session S6 as an example.

Although the saved .pkl.gz file already includes the final LFADS rates, inferred inputs, and initial condition, we reconstruct the intermediate steps here for reproducibility and understanding.

**Note:**  
This notebook takes as input a pre-filtered dataset saved in a gzip-compressed pickle file published as the associated dataset to the paper *"The geometry of the neural state space of decisions"* in [Zenodo](https://zenodo.org/records/15093134). The dataset originates from the raw recordings available at [Zenodo](https://zenodo.org/records/13207505), but includes only completed **decision trials** (`trialType==20`), excluding other tasks or incomplete trials.

After generating the LFADS training dataset, we load the results of running AutoLFADS and augment the DataFrame with several new columns:

- `'SpikeCount'`: Trial-aligned spike count matrix used for training the LFADS model (firing rates in Hz, 10 ms bins) with shape `(Neurons × Time)`.
- `'LFADS'`: Smoothed firing rates inferred by AutoLFADS for the same neurons and time windows with shape `(Neurons × Time)`.
- `'InferredInput'`: Causal LFADS-inferred input for each trial, stored as an array of shape `(3 × Time)`.
- `'InitialCondition'`: The inferred generator initial condition for each trial, stored as a vector corresponding to the latent generator state at trial onset.

Time is cropped from dot-motion onset to saccade completion — so trial durations vary (not padded with zeros).

##
## Step 0: Set root directory and download some data

In [ ]:
import os
import sys

# Automatically find the project root (directory containing 'src' or 'data')
def find_project_root(marker_dirs=("src", "data","notebooks")):
    path = os.getcwd()
    while path != "/" and not all(os.path.exists(os.path.join(path, d)) for d in marker_dirs):
        path = os.path.dirname(path)
    return path

PROJECT_ROOT = find_project_root()

# Set up Python import path and working directory
sys.path.append(os.path.join(PROJECT_ROOT, "src"))
os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)


In [ ]:

from src.io_utils import download_session

# Download and get the path to the file
# Also sets the working session

session = "S8"
path = download_session(session)



##
## Step 1: Load preprocessed dataset

In [ ]:

from src.io_utils import load_dataframe_with_metadata

df = load_dataframe_with_metadata(session)

print(f"Session: {df.attrs['Session']}, Monkey: {df.attrs['Monkey']}, Date: {df.attrs['Date'].date()}")
print(f"Number of neurons: {df.attrs['NCells']}")
print(f"Columns: {df.columns.tolist()}")
df.head()


##
## Step 2: Construct training and validation datasets for AutoLFADS

We split the dataset into validation and training, mantaining a choice and coherence balanced split.

In [ ]:

import numpy as np
import h5py

coherences = np.sort(df['coh'].unique())
choices = np.sort(df['choice'].unique())
train_idx, valid_idx = [], []

for choice in choices:
    for coh in coherences:
        idx = df[(df['choice'] == choice) & (df['coh'] == coh)].index.tolist()
        np.random.shuffle(idx)
        split = int(2 / 3 * len(idx))
        train_idx.extend(idx[:split])
        valid_idx.extend(idx[split:])

train_idx, valid_idx = np.array(train_idx), np.array(valid_idx)



We create 10ms-binned spike count data to train the LFADS model

In [ ]:
bin_size_ms = 10
bin_size_s = bin_size_ms / 1000

max_duration = np.max(df['saccadeComplete'] - df['dotsOn'])
bins = np.arange(0, max_duration + bin_size_s, bin_size_s)
times = bins[:-1] + bin_size_s / 2  # optional, kept if you use it elsewhere


def process_trial(trial, df, bins, N):
    """
    Process a single trial to generate only the spike array.

    Parameters:
        trial: int
            Index of the trial to process.
        df: pandas.DataFrame
            DataFrame containing trial data and spike times.
        bins: np.ndarray
            Array of bin edges for spike binning (relative to dotsOn).
        N: int
            Number of neurons to include.

    Returns:
        spike_array: np.ndarray
            Array of shape (time, neurons) with binned spike counts,
            aligned to dotsOn (t0) and binned out to max_duration.
    """
    t0 = df['dotsOn'].loc[trial]

    # df['spCellPop'].loc[trial] is assumed to be a list-like of length >= N,
    # where each entry is an array/list of spike times in absolute seconds.
    spikes_binned = [
        np.histogram(np.asarray(sp) - t0, bins=bins)[0]
        for sp in df['spCellPop'].loc[trial][:N]
    ]
    spike_array = np.stack(spikes_binned, axis=-1)  # (time, neurons)

    return spike_array


NCells = df.attrs['NCells']

DataT = np.stack([process_trial(trial, df, bins, NCells) for trial in train_idx], axis=0)
DataV = np.stack([process_trial(trial, df, bins, NCells) for trial in valid_idx], axis=0)

# DataT.shape == (n_train_trials, time, neurons)
# DataV.shape == (n_valid_trials, time, neurons)

##
## Step 3: Save HDF5 file for AutoLFADS


In [ ]:
# For local AutoLFADS training

filename = f"DataLIP_10ms_{session}.h5"
with h5py.File(filename, 'w') as hf:
    hf.create_dataset('train_data', data=DataT)
    hf.create_dataset('valid_data', data=DataV)
    hf.create_dataset('IndT', data=train_idx)
    hf.create_dataset('IndV', data=valid_idx)

print(f"Saved LFADS dataset to {filename}")

##
## Step 4: Running AutoLFADS

We use the preprocessed dataset `DataLIP_10ms_S6.h5` together with the config file `S6.yaml`.
This YAML file specifies all the training parameters, architecture details, and hyperparameter search settings. YAML files used for all sessions are found in the `data` folder.

📖 How AutoLFADS works:
AutoLFADS (Keshtkaran et al., 2022) is a deep learning framework based on LFADS (Pandarinath et al., 2018) that uses population-based training (PBT) to automatically tune model hyperparameters.
It learns latent dynamical structure from neural spike trains to infer smooth firing rates using variational autoencoders.

**References:**  
- AutoLFADS paper: [Keshtkaran et al., 2022](https://doi.org/10.1038/s41592-022-01675-0)  
- Original LFADS paper: [Pandarinath et al., 2018](https://doi.org/10.1038/s41592-018-0109-9)


##
## Step 5: Load LFADS results and populate DataFrame

> **Note**: The next cell requires the results from running AutoLFADS in Step 4, not included in the dataset.

In [ ]:
import h5py

bin_size_ms = 10

# Load outputs made locally with AutoLFADS
lfads_path = f'../Causal_3DInput/{session}/posterior_samples.h5'

with h5py.File(lfads_path, 'r') as data:
    initial_condition_train, initial_condition_valid = data['train_ic_post_mean'][:], data['valid_ic_post_mean'][:]
    lfads_train, lfads_valid = data['train_rates'][:], data['valid_rates'][:]
    inferred_inputs_train, inferred_inputs_valid = data['train_gen_inputs'][:], data['valid_gen_inputs'][:]

# Load the data used for training
data_path = f'../Causal_3DInput/DataLIP_10ms_{session}.h5'

with h5py.File(data_path, "r") as data:
    IndT, IndV = data['IndT'][:], data['IndV'][:]
    data_train, data_valid = data['train_data'][:], data['valid_data'][:]


amp = 1000/bin_size_ms # to get firing rates
df['LFADS'], df['SpikeCount'], df['InferredInput'], df['InitialCondition'] = None, None, None, None

for i, trial in enumerate(IndT):
    t_end = int(100 * (df['saccadeDetected'].loc[trial] - df['dotsOn'].loc[trial]))
    df.at[trial, 'LFADS'] = (amp * lfads_train[i][:t_end]).T
    df.at[trial, 'SpikeCount'] = (amp * data_train[i][:t_end]).T
    df.at[trial, 'InferredInput'] =  (inferred_inputs_train[i][:t_end]).T
    df.at[trial, 'InitialCondition'] = amp* initial_condition_train[i]

for i, trial in enumerate(IndV):
    t_end = int(100 * (df['saccadeDetected'].loc[trial] - df['dotsOn'].loc[trial]))
    df.at[trial, 'LFADS'] = (amp * lfads_valid[i][:t_end]).T
    df.at[trial, 'SpikeCount'] = (amp * data_valid[i][:t_end]).T
    df.at[trial, 'InferredInput'] =  (inferred_inputs_valid[i][:t_end]).T
    df.at[trial, 'InitialCondition'] = amp* initial_condition_valid[i]

Save the final dataframe with session metadata

In [ ]:
from src.io_utils import save_dataframe_with_metadata

save_dataframe_with_metadata(df, session)